# Risk-Adjusted Business Deal Valuation: Two-Stage Heavy-Tail Pipeline
### Development Task: End-to-End EDA & Robust Modeling Framework for Sparse, Severe Losses

---

## Part 1: Briefing for Colleagues — Executive Data Quality & Generation Observations

### Data-Quality and Sparsity Diagnostics
* **`loss_amount_usd` (38 Missing / High Sparsity):** This column is highly sparse. Out of the rows where a loss is explicitly checked (`loss_recorded == 1`), 38 rows are completely unrecorded. In a real-world scenario, this represents an operational leak—losses occurred, but the accounting ledger failed to log the absolute valuation. Imputing these with `0` is toxic because it trains our model that these risk patterns result in zero financial harm. We must drop these 38 rows to preserve target integrity.
* **`days_to_loss_recorded` (1190 Missing):** This missingness is **structural**, not an error. The values are empty precisely because no loss occurred (`loss_recorded == 0`). Including this as an input feature for classification induces an immediate **data leak** (the model reads the presence of a number as proof a loss will happen). It must be excluded from Phase 1 and used exclusively in Phase 2 conditionally.
* **`notes` (1241 Missing):** Free-form string fields break downstream models. However, the *presence* of metadata denotes an administrative or informal flag. We map this into a binary indicator feature: `has_notes` (0 or 1).
* **`buyer_region` (73 Missing):** Randomly unrecorded categorical feature. To avoid introducing geographical assignment bias through mode-imputation, we assign these records to a distinct `'Unknown'` bucket.

### Synthetic Data Artifacts & Trust Vector
The uniform layout of missing records strongly indicates this dataset was simulated using a classic conditional mask (e.g., $P(\text{Missing} \mid \text{Loss}=0) = 1.0$). While functional for constructing pipelines, synthetic tail distributions often fail to mimic true macroeconomic correlation clusters. Our model might over-index on isolated features like `route_risk_flag` or `rep_confidence_score` without learning real-world systemic shock patterns.

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, auc, f1_score, classification_report, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, HuberRegressor, Ridge
import xgboost as xgb

import warnings
warnings.filterwarnings('ignore')

print("All standard financial risk libraries successfully imported.")

All standard financial risk libraries successfully imported.


### Data Pipeline Setup
We will synthesize a structural mock dataset mapping your precise schema footprints to guarantee full execution out of the box.

In [10]:
np.random.seed(42)
n_samples = 1500

data = {
    'order_id': [f"ORD_{i:04d}" for i in range(n_samples)],
    'ship_date': pd.date_range(start='2024-01-01', periods=n_samples, freq='h'),
    'buyer_region': np.random.choice(['North', 'South', 'East', 'West', np.nan], size=n_samples, p=[0.4, 0.3, 0.15, 0.1, 0.05]),
    'product_category': np.random.choice(['Electronics', 'Industrial', 'Chemicals', 'Textiles'], size=n_samples),
    'order_value_usd': np.random.exponential(scale=15000, size=n_samples) + 2000,
    'buyer_tenure_yrs': np.random.randint(0, 12, size=n_samples),
    'new_buyer': np.random.choice([0, 1], size=n_samples, p=[0.8, 0.2]),
    'payment_terms_days': np.random.choice([15, 30, 45, 60], size=n_samples),
    'route_risk_flag': np.random.choice([0, 1], size=n_samples, p=[0.92, 0.08]),
    'rep_confidence_score': np.random.uniform(50, 100, size=n_samples),
}

df_mock = pd.DataFrame(data)

# Constructing sparse operational loss triggers
df_mock['loss_recorded'] = 0
loss_idx = df_mock.sample(frac=0.18, random_state=42).index
df_mock.loc[loss_idx, 'loss_recorded'] = 1

# Simulating heavy-tailed Pareto distributions for loss value sizing
df_mock['loss_amount_usd'] = np.nan
base_loss = (np.random.pareto(a=1.2, size=len(loss_idx)) + 1) * 8000
df_mock.loc[loss_idx, 'loss_amount_usd'] = base_loss

# Creating target corrupted records (38 rows missing target value)
corrupt_loss_idx = df_mock[df_mock['loss_recorded'] == 1].sample(n=38, random_state=42).index
df_mock.loc[corrupt_loss_idx, 'loss_amount_usd'] = np.nan

# Injecting structural missing metrics to mimic real schema boundaries
df_mock['days_to_loss_recorded'] = np.nan
valid_loss_idx = df_mock[(df_mock['loss_recorded'] == 1) & (df_mock['loss_amount_usd'].notnull())].index
df_mock.loc[valid_loss_idx, 'days_to_loss_recorded'] = np.random.randint(5, 45, size=len(valid_loss_idx))

df_mock['notes'] = np.nan
df_mock.loc[df_mock.sample(n=259, random_state=42).index, 'notes'] = "Risk warning or delivery discrepancy flags."

print("Shape of structural raw dataset:", df_mock.shape)
print(df_mock.isnull().sum())

Shape of structural raw dataset: (1500, 14)
order_id                    0
ship_date                   0
buyer_region                0
product_category            0
order_value_usd             0
buyer_tenure_yrs            0
new_buyer                   0
payment_terms_days          0
route_risk_flag             0
rep_confidence_score        0
loss_recorded               0
loss_amount_usd          1268
days_to_loss_recorded    1268
notes                    1241
dtype: int64


### Data Cleansing Framework
We now clean the schema deterministically based on our strategy outline.

In [11]:
# 1. Drop corrupted target instances
corrupted_condition = (df_mock['loss_recorded'] == 1) & (df_mock['loss_amount_usd'].isnull())
df_clean = df_mock[~corrupted_condition].copy()

# 2. Transform unstructured categorical text fields into highly responsive numerical triggers
df_clean['has_notes'] = df_clean['notes'].notnull().astype(int)
df_clean.drop(columns=['notes'], inplace=True, errors='ignore')

# 3. Unify undefined regional factors
df_clean['buyer_region'] = df_clean['buyer_region'].fillna('Unknown')

print("Cleaned Dataset Structural State:", df_clean.shape)
print(df_clean.isnull().sum())

Cleaned Dataset Structural State: (1462, 14)
order_id                    0
ship_date                   0
buyer_region                0
product_category            0
order_value_usd             0
buyer_tenure_yrs            0
new_buyer                   0
payment_terms_days          0
route_risk_flag             0
rep_confidence_score        0
loss_recorded               0
loss_amount_usd          1230
days_to_loss_recorded    1230
has_notes                   0
dtype: int64


## Part 2: Two-Stage Hurdle Model Compilation

### Modeling Choice & Justification
A direct single-stage regression model targeting `loss_amount_usd` will entirely fail because over 80% of the dataset features a true loss of $0. The model would aggressively skew towards zero, underestimating severe exposures.

**Our Solution:** A **Two-Stage Hurdle Framework**:
1. **Stage 1 (Classification Model):** Estimates $P(\text{Loss}=1 \mid X)$. Evaluated using **PR-AUC** to protect the company against severe class imbalances.
2. **Stage 2 (Heavy-Tail Robust Regression Model):** Trained *only* on sub-segments where a loss was realized. We employ log-transformed labels optimized through robust algorithms like **Huber Loss** to prevent severe outliers from distorting convergence parameters.

In [12]:
# Feature Matrix Formatting
categorical_cols = ['buyer_region', 'product_category']
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

# Ensure leak targets are fully stripped from features
exclude_features = ['order_id', 'ship_date', 'loss_recorded', 'loss_amount_usd', 'days_to_loss_recorded']
feature_cols = [c for c in df_encoded.columns if c not in exclude_features]

X = df_encoded[feature_cols]
y_class = df_encoded['loss_recorded']

X_train, X_test, y_train_cls, y_test_cls = train_test_split(X, y_class, test_size=0.2, random_state=42, stratify=y_class)

print(f"Training set shape: {X_train.shape}, Imbalance distribution: {np.bincount(y_train_cls)}")

Training set shape: (1169, 14), Imbalance distribution: [983 186]


### Stage 1: Evaluating Multiple Classifiers via PR-AUC and F1-Score

In [13]:
models_cls = {
    "Logistic Regression (Balanced)": LogisticRegression(class_weight='balanced', max_iter=1000),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "XGBoost Classifier": xgb.XGBClassifier(scale_pos_weight=(len(y_train_cls)-sum(y_train_cls))/sum(y_train_cls), eval_metric='logloss', random_state=42)
}

for name, model in models_cls.items():
    model.fit(X_train, y_train_cls)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    
    precision, recall, _ = precision_recall_curve(y_test_cls, probs)
    pr_auc = auc(recall, precision)
    f1 = f1_score(y_test_cls, preds)
    
    print(f"=== {name} ===")
    print(f"PR-AUC Score: {pr_auc:.4f} | Max Balanced F1-Score: {f1:.4f}")
    print(classification_report(y_test_cls, preds))
    print("\n")

=== Logistic Regression (Balanced) ===
PR-AUC Score: 0.9840 | Max Balanced F1-Score: 0.9890
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       247
           1       1.00      0.98      0.99        46

    accuracy                           1.00       293
   macro avg       1.00      0.99      0.99       293
weighted avg       1.00      1.00      1.00       293



=== Random Forest Classifier ===
PR-AUC Score: 0.9863 | Max Balanced F1-Score: 0.9890
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       247
           1       1.00      0.98      0.99        46

    accuracy                           1.00       293
   macro avg       1.00      0.99      0.99       293
weighted avg       1.00      1.00      1.00       293



=== XGBoost Classifier ===
PR-AUC Score: 0.9827 | Max Balanced F1-Score: 0.9890
              precision    recall  f1-score   support

           0       1.00      

### Stage 2: Heavy-Tail Severity Extrapolation Regression

In [14]:
# Filter down strictly to active loss instances
df_losses = df_encoded[df_encoded['loss_recorded'] == 1]
X_reg = df_losses[feature_cols]
y_reg = df_losses['loss_amount_usd']

# Mitigate extreme variance via natural logarithmic compression
y_reg_log = np.log1p(y_reg)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg_log, test_size=0.2, random_state=42)

models_reg = {
    "Standard Ridge Regression": Ridge(),
    "Huber Robust Regressor": HuberRegressor(max_iter=2000)
}

for name, model in models_reg.items():
    model.fit(X_train_reg, y_train_reg)
    log_preds = model.predict(X_test_reg)
    
    # Inverse transform to compute metrics in true USD terms
    true_preds = np.expm1(log_preds)
    true_target = np.expm1(y_test_reg)
    
    mae = mean_absolute_error(true_target, true_preds)
    r2 = r2_score(true_target, true_preds)
    
    print(f"=== {name} ===")
    print(f"True Dollar Metric Mean Absolute Error: ${mae:,.2f}")
    print(f"R2 Valuation Stability Metric: {r2:.4f}\n")

=== Standard Ridge Regression ===
True Dollar Metric Mean Absolute Error: $27,897.13
R2 Valuation Stability Metric: -0.1139

=== Huber Robust Regressor ===
True Dollar Metric Mean Absolute Error: $29,608.71
R2 Valuation Stability Metric: -0.1505



### Model Limitations and Failure Modes
1. **Out-of-Sample Violations:** If a customer encounters an unprecedented default size ($10M+), the log-linear or Huber models will systematically under-predict the severity because they lack structural context regarding concentration limitations.
2. **Information Asymmetry:** The model treats risk uniformly inside a region. It cannot dynamically sense rapid deterioration in localized counterparties without real-time external telemetry features.